## Pydantic을 이용해 Tool의 입력값 정의

In [6]:
from langchain_core.tools import tool
import yfinance as yf
import pprint
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True,
)

llm_with_tools = llm.bind_tools(tools)

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

## Pydantic으로 정의한 입력값을 Tool에 사용 / Tool을 LLM에 등록

In [7]:
@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_yf_stock_history]
tool_dict = {"get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

## 사용자의 질문을 LLM에 전달

In [11]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.")
]

messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)


print("=== 응답 ===")
print(response.content)

print("=== 툴 콜 ===")
print(response.tool_calls)

content=[{'id': 'rs_07023c1d8e16d361006a9d618634dc87d09e4c01189cd02792', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqnWGG5d66TvCHLOPm2D-3-fjyQ3fGK9TlJ1z5s722JNJPtVbxh8LyprZxz5kvjd3FLWd1EodxCXfnfUbyzsjz1EMhN2qY_3BFop-dVkEMvQL4OFEbYwxF_32kI8muiaISwIUjg2857ky4qvxiLjqDo1wie5uUV-nCANTL_NKJLEAXhmQ-nHcvA2xXtW_p44HT7OqZ2BAaquzgFMfWU9f3fcq0dy_Tu9UkDsqRGTKlqd3abNkd1Gg1yKcSQ5iVXyAtU1l2E2Orw2upP-FHoTWD3wUg4-Y6t8yZ_EiZ260oT2Zd8t3K8x0J3dKS4Gp3eiy6m30memyOekM_AEUsHt5j8MeIiZICQmLr0vSZBURelYfCoJ3yqBDZqSMHtsyy6GcNoP_oiMkwqhzT4y_NUbJLDhrwy3QSTTXzxmcPFtc__50vJtDOAVOtkgiaj6EtL5KhEa_1meoyTtVeYdaYaHWT-L4686eOG3fSfpIZd6AnYLRIra11yZdW_rl0ULQXiIIczSHe0EMXSHlbdj0-vmaxaOFZN7q3YFybaROI_Bk8VkR_LNpUwC_8PiBFTeYsXIxZ02_Za2xwhopyZNImg7qxSVTmnmNC9PSNXzneWfJs5HLLNA4_I7zSAVxYMt2YYfvdvjV_bHmZfnwTKDdH6zeIHu2KrBTDPowHXNsZh0rCUlwBrXaHNnei5jZ2022UJvND4N3H1kUasuNizjIfpHRDyUcalW1zUmZjcOnAGg_CI-i6_XNm2xUD5t7eqaCtWqVulleALcKGQNhGr7DC1UgmC1bDCxNlEKOZiDH_ycKKG4zp2jBNx0Gkh6KwV64V-oX57A81z5ja9FIX7I

- tool_calls가 get_yf_stock_history로 나온 것 = 툴 선택 성공

## LLM이 선택한 Tool 실행

In [12]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

print("=== 툴 결과 ===")
print(tool_msg.content)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-08-05 00:00:00-04:00 | 323.42 | 327.14 | 320.28 |  321.55 | 2.78208e+07 |           0 |              0 |\n| 2026-08-06 00:00:00-04:00 | 317.05 | 323    | 315.52 |  319.53 | 2.60338e+07 |           0 |              0 |\n| 2026-08-07 00:00:00-04:00 | 322.34 | 333.73 | 321.25 |  328.58 | 3.94923e+07 |           0 |              0 |\n| 2026-08-10 00:00:00-04:00 | 326.6  | 332.05 | 326.15 |  330.88 | 2.50038e+07 |           0 |              0 |\n| 2026-08-11 00:00:00-04:00 | 332.8  | 336.2  | 329.53 |  332.81 | 2.33452e+07 |           0 |              0 |\n| 2026-08-12 00:00:00-04:00 | 335    | 335.5  | 323.64 |  327.51 | 2.86989e+07 |           0 |              0 |\n| 2026-08-13 00:00:00-04:0

- selected_tool = ... → GPT가 선택한 함수(get_yf_stock_history)를 찾음
- selected_tool.invoke(tool_call) → 그 함수에 GPT가 만든 인자를 넣어 실제 실행함
- messages.append(tool_msg) → 실행 결과를 대화 기록에 넣어서, 다음 LLM 호출이 결과를 보고 최종 답변하도록 준비함

## Tool의 결과를 LLM에게 전달 / 최종 답변 생성

In [14]:
response = llm_with_tools.invoke(messages)

final_answer = next(
    item["text"]
    for item in response.content
    if item.get("type") == "text"
)

print(final_answer)

테슬라(TSLA)는 **한 달 전보다 올랐습니다.**

- 한 달 전 종가: **321.55달러** (8월 5일)
- 최근 종가: **354.08달러** (9월 4일)
- 변동: **약 10.1% 상승**  

※ 조회된 최근 거래일 기준이며, 장중 가격에 따라 달라질 수 있습니다.


```txt
👤 사용자
   │
   │ "테슬라 한 달 전보다 올랐나?"
   ▼
🤖 LLM
   │
   │ tool_call 생성
   │ get_yf_stock_history
   │ ticker=TSLA, period=1mo
   ▼
🐍 Python
   │
   │ selected_tool.invoke(tool_call)
   ▼
📈 yfinance
   │
   │ 주가 데이터 조회
   ▼
📦 ToolMessage
   │
   │ messages.append(tool_msg)
   ▼
🤖 LLM
   │
   │ llm_with_tools.invoke(messages)
   ▼
💬 최종 답변
   "테슬라는 한 달 전보다 약 10.1% 올랐습니다."
```